<a href="https://colab.research.google.com/github/Mridul33/capstone-project/blob/main/Stage_6_Knowledge_Graph_Construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd

ner_path = (
    "/content/drive/MyDrive/Capstone_Project/"
    "Stage_3_1_Outputs/pubmedbert_ner_test_predictions.pkl"
)

re_path = (
    "/content/drive/MyDrive/Capstone_Project/"
    "Stage_3_3_Outputs/pubmedbert_re_test_predictions.pkl"
)

ner_df = pd.read_pickle(ner_path)
re_df = pd.read_pickle(re_path)

print("NER predictions:", len(ner_df))
print("RE predictions:", len(re_df))

display(ner_df.head())
display(re_df.head())

NER predictions: 3681
RE predictions: 2300


,document_id,chunk_id,entity_text,entity_type
0,15485686,0,scn5a,GeneOrGeneProduct
1,15485686,0,long qt syndrome,DiseaseOrPhenotypicFeature
2,15485686,0,tachycardia /,DiseaseOrPhenotypicFeature
3,15485686,0,bradycardia,DiseaseOrPhenotypicFeature
4,15485686,0,congenital long qt syndrome,DiseaseOrPhenotypicFeature


,document_id,entity1_identifier,entity1_text,entity1_type,entity2_identifier,entity2_text,entity2_type,gold_relation,predicted_relation
0,15485686,D001919,bradycardia,DiseaseOrPhenotypicFeature,6331,SCN5A,GeneOrGeneProduct,Association,Association
1,15485686,D001919,bradycardia,DiseaseOrPhenotypicFeature,p|SUB|V|1763|M,V1763M,SequenceVariant,Positive_Correlation,Positive_Correlation
2,15485686,D013610,tachycardia,DiseaseOrPhenotypicFeature,6331,SCN5A,GeneOrGeneProduct,Association,Association
3,15485686,D013610,tachycardia,DiseaseOrPhenotypicFeature,p|SUB|V|1763|M,V1763M,SequenceVariant,Positive_Correlation,Association
4,15485686,6331,SCN5A,GeneOrGeneProduct,D001145,arrhythmias,DiseaseOrPhenotypicFeature,Association,Association


In [ ]:
# Build candidate entity table from both RE entity columns

entities_1 = re_df[[
    "entity1_identifier",
    "entity1_text",
    "entity1_type"
]].rename(columns={
    "entity1_identifier": "entity_id",
    "entity1_text": "entity_name",
    "entity1_type": "entity_type"
})

entities_2 = re_df[[
    "entity2_identifier",
    "entity2_text",
    "entity2_type"
]].rename(columns={
    "entity2_identifier": "entity_id",
    "entity2_text": "entity_name",
    "entity2_type": "entity_type"
})

entities_df = pd.concat(
    [entities_1, entities_2],
    ignore_index=True
)

print("Total entity occurrences:", len(entities_df))
print("Unique identifiers:", entities_df["entity_id"].nunique())

display(entities_df.head(10))

Total entity occurrences: 4600
Unique identifiers: 949


,entity_id,entity_name,entity_type
0,D001919,bradycardia,DiseaseOrPhenotypicFeature
1,D001919,bradycardia,DiseaseOrPhenotypicFeature
2,D013610,tachycardia,DiseaseOrPhenotypicFeature
3,D013610,tachycardia,DiseaseOrPhenotypicFeature
4,6331,SCN5A,GeneOrGeneProduct
5,D001145,arrhythmias,DiseaseOrPhenotypicFeature
6,D001145,arrhythmias,DiseaseOrPhenotypicFeature
7,p|SUB|V|1763|M,V1763M,SequenceVariant
8,p|SUB|V|1763|M,V1763M,SequenceVariant
9,D008133,long QT syndrome,DiseaseOrPhenotypicFeature


In [ ]:
# Check whether one identifier appears with multiple names or types

name_conflicts = (
    entities_df.groupby("entity_id")["entity_name"]
    .nunique()
    .sort_values(ascending=False)
)

type_conflicts = (
    entities_df.groupby("entity_id")["entity_type"]
    .nunique()
    .sort_values(ascending=False)
)

print("Identifiers with multiple names:")
display(name_conflicts[name_conflicts > 1].head(20))

print("\nIdentifiers with multiple entity types:")
display(type_conflicts[type_conflicts > 1])

Identifiers with multiple names:


,entity_name
entity_id,
D009369,9
9606,7
D007674,6
D003920,4
D007249,4
D020258,3
10090,3
D008607,3
D008659,3



Identifiers with multiple entity types:


,entity_type
entity_id,
-,2


In [ ]:
# Remove invalid / missing identifiers

clean_entities = entities_df[
    entities_df["entity_id"].notna()
].copy()

clean_entities = clean_entities[
    clean_entities["entity_id"].astype(str).str.strip() != "-"
].copy()

print("Entity occurrences after removing '-' IDs:",
      len(clean_entities))

Entity occurrences after removing '-' IDs: 4593


In [ ]:
# Choose the most frequent name and type for each biomedical identifier

canonical_entities = (
    clean_entities
    .groupby(["entity_id", "entity_name", "entity_type"])
    .size()
    .reset_index(name="frequency")
)

canonical_entities = (
    canonical_entities
    .sort_values(
        ["entity_id", "frequency"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset=["entity_id"],
        keep="first"
    )
    .reset_index(drop=True)
)

canonical_entities = canonical_entities[
    ["entity_id", "entity_name", "entity_type"]
]

print("Final unique KG entities:",
      len(canonical_entities))

display(canonical_entities.head(20))

Final unique KG entities: 948


,entity_id,entity_name,entity_type
0,100006951,Neuregulin 2a,GeneOrGeneProduct
1,10029,Chinese hamster,OrganismTaxon
2,100770962,OPRM1,GeneOrGeneProduct
3,10090,mice,OrganismTaxon
4,10116,rat,OrganismTaxon
5,10135,visfatin,GeneOrGeneProduct
6,1017,CDK2 [p33],GeneOrGeneProduct
7,1019,CDK4,GeneOrGeneProduct
8,101910198,AKT,GeneOrGeneProduct
9,1021,CDK6,GeneOrGeneProduct


In [ ]:
print("Duplicate entity IDs:",
      canonical_entities["entity_id"].duplicated().sum())

print("\nEntity type distribution:")
print(
    canonical_entities["entity_type"]
    .value_counts()
)

Duplicate entity IDs: 0

Entity type distribution:
entity_type
GeneOrGeneProduct             395
DiseaseOrPhenotypicFeature    227
ChemicalEntity                163
SequenceVariant               135
CellLine                       17
OrganismTaxon                  11
Name: count, dtype: int64


In [ ]:
# Build graph relationships from PubMedBERT predicted relations

relations_df = re_df[
    re_df["predicted_relation"] != "No_Relation"
].copy()

# Remove unusable identifiers
relations_df = relations_df[
    relations_df["entity1_identifier"].astype(str).str.strip() != "-"
]

relations_df = relations_df[
    relations_df["entity2_identifier"].astype(str).str.strip() != "-"
]

relations_df = relations_df[[
    "document_id",
    "entity1_identifier",
    "entity2_identifier",
    "predicted_relation"
]].rename(columns={
    "entity1_identifier": "source_id",
    "entity2_identifier": "target_id",
    "predicted_relation": "relation_type"
})

relations_df = relations_df.reset_index(drop=True)

print("Final KG relationships:",
      len(relations_df))

display(relations_df.head(20))

Final KG relationships: 1126


,document_id,source_id,target_id,relation_type
0,15485686,D001919,6331,Association
1,15485686,D001919,p|SUB|V|1763|M,Positive_Correlation
2,15485686,D013610,6331,Association
3,15485686,D013610,p|SUB|V|1763|M,Association
4,15485686,6331,D001145,Association
5,15485686,D001145,D008801,Positive_Correlation
6,15485686,D001145,D008012,Positive_Correlation
7,15485686,p|SUB|V|1763|M,D001145,Association
8,15485686,p|SUB|V|1763|M,D008133,Positive_Correlation
9,15485686,D008133,6331,Association


In [ ]:
print("Relation distribution:")
print(
    relations_df["relation_type"]
    .value_counts()
)

print("\nMissing source IDs:",
      (~relations_df["source_id"]
       .isin(canonical_entities["entity_id"]))
      .sum())

print("Missing target IDs:",
      (~relations_df["target_id"]
       .isin(canonical_entities["entity_id"]))
      .sum())

Relation distribution:
relation_type
Association             491
Positive_Correlation    451
Negative_Correlation    156
Comparison               12
Cotreatment              11
Bind                      5
Name: count, dtype: int64

Missing source IDs: 0
Missing target IDs: 0


In [ ]:
import os

kg_save_directory = (
    "/content/drive/MyDrive/Capstone_Project/"
    "Stage_6_Knowledge_Graph"
)

os.makedirs(kg_save_directory, exist_ok=True)

# Making sure that IDs stay as strings
canonical_entities["entity_id"] = (
    canonical_entities["entity_id"].astype(str)
)

relations_df["source_id"] = (
    relations_df["source_id"].astype(str)
)

relations_df["target_id"] = (
    relations_df["target_id"].astype(str)
)

entities_path = os.path.join(
    kg_save_directory,
    "entities.csv"
)

relations_path = os.path.join(
    kg_save_directory,
    "relations.csv"
)

canonical_entities.to_csv(
    entities_path,
    index=False
)

relations_df.to_csv(
    relations_path,
    index=False
)

print("Saved entities:", entities_path)
print("Saved relations:", relations_path)

print("\nEntities:", len(canonical_entities))
print("Relations:", len(relations_df))

Saved entities: /content/drive/MyDrive/Capstone_Project/Stage_6_Knowledge_Graph/entities.csv
Saved relations: /content/drive/MyDrive/Capstone_Project/Stage_6_Knowledge_Graph/relations.csv

Entities: 948
Relations: 1126


In [ ]:
print(os.listdir(kg_save_directory))

print("\nENTITIES:")
display(pd.read_csv(entities_path, dtype=str).head())

print("\nRELATIONS:")
display(pd.read_csv(relations_path, dtype=str).head())

['entities.csv', 'relations.csv']

ENTITIES:


,entity_id,entity_name,entity_type
0,100006951,Neuregulin 2a,GeneOrGeneProduct
1,10029,Chinese hamster,OrganismTaxon
2,100770962,OPRM1,GeneOrGeneProduct
3,10090,mice,OrganismTaxon
4,10116,rat,OrganismTaxon



RELATIONS:


,document_id,source_id,target_id,relation_type
0,15485686,D001919,6331,Association
1,15485686,D001919,p|SUB|V|1763|M,Positive_Correlation
2,15485686,D013610,6331,Association
3,15485686,D013610,p|SUB|V|1763|M,Association
4,15485686,6331,D001145,Association


In [ ]:
!pip install -q neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 7.5 MB/s eta 0:00:00


In [ ]:
from neo4j import GraphDatabase

NEO4J_URI = "neo4j+s://a1848d18.databases.neo4j.io"
NEO4J_USERNAME = "a1848d18"
NEO4J_PASSWORD = "gGTJpOvR0np1sTlCLBmffl-JuUCq3uaM4vnkEc5D5Hw"

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j Aura successfully.")

Connected to Neo4j Aura successfully.


In [ ]:
constraint_query = """
CREATE CONSTRAINT entity_id_unique IF NOT EXISTS
FOR (e:Entity)
REQUIRE e.entity_id IS UNIQUE
"""

driver.execute_query(constraint_query)

print("Entity ID uniqueness constraint created.")

Entity ID uniqueness constraint created.


In [ ]:
# Convert dataframe rows to dictionaries
entity_records = canonical_entities[
    ["entity_id", "entity_name", "entity_type"]
].to_dict("records")

print("Entities to upload:", len(entity_records))

entity_query = """
UNWIND $entities AS row

MERGE (e:Entity {entity_id: row.entity_id})

SET e.name = row.entity_name,
    e.type = row.entity_type
"""

result = driver.execute_query(
    entity_query,
    entities=entity_records
)

print("Entity upload completed.")
print(result.summary.counters)

Entities to upload: 948
Entity upload completed.
SummaryCounters{labels_added: 948, nodes_created: 948, properties_set: 2844, contains_updates: True, contains_system_updates: False}


In [ ]:
records, summary, keys = driver.execute_query("""
MATCH (e:Entity)
RETURN count(e) AS entity_count
""")

print("Entities in Neo4j:", records[0]["entity_count"])

Entities in Neo4j: 948


In [ ]:
duplicate_edges = relations_df.duplicated(
    subset=[
        "document_id",
        "source_id",
        "target_id",
        "relation_type"
    ]
).sum()

print("Relationship rows:", len(relations_df))
print("Exact duplicate relationships:", duplicate_edges)

Relationship rows: 1126
Exact duplicate relationships: 0


In [ ]:
import re

relationship_records = relations_df.to_dict("records")

print("Relationships to upload:", len(relationship_records))

for relation_type in relations_df["relation_type"].unique():

    # Safety check for Neo4j relationship type
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", relation_type)

    rows = [
        row for row in relationship_records
        if row["relation_type"] == relation_type
    ]

    query = f"""
    UNWIND $rows AS row

    MATCH (source:Entity {{entity_id: row.source_id}})
    MATCH (target:Entity {{entity_id: row.target_id}})

    MERGE (source)-[r:{relation_type} {{
        document_id: row.document_id
    }}]->(target)

    SET r.model = "PubMedBERT"
    """

    driver.execute_query(
        query,
        rows=rows
    )

    print(
        relation_type,
        "uploaded:",
        len(rows)
    )

print("Relationship upload completed.")

Relationships to upload: 1126
Association uploaded: 491
Positive_Correlation uploaded: 451
Negative_Correlation uploaded: 156
Comparison uploaded: 12
Cotreatment uploaded: 11
Bind uploaded: 5
Relationship upload completed.


In [ ]:
records, summary, keys = driver.execute_query("""
MATCH ()-[r]->()
RETURN count(r) AS relationship_count
""")

print(
    "Relationships in Neo4j:",
    records[0]["relationship_count"]
)

Relationships in Neo4j: 1127


In [ ]:
records, summary, keys = driver.execute_query("""
MATCH ()-[r]->()
WHERE r.model = "PubMedBERT"
RETURN count(r) AS pubmedbert_relationships
""")

print(
    "PubMedBERT relationships:",
    records[0]["pubmedbert_relationships"]
)

PubMedBERT relationships: 1126
